# #3 Plot UMAP embeddings

## Purpose

Generate plots using UMAP embedding generated in step 2.

## Setup

In [188]:
import scanpy as sc
import treedata as td
import numpy as np
from cycler import cycler
import pandas as pd
import matplotlib.pyplot as plt

from devmap.config import set_theme, get_paths, subtype_palette, celltype_palette, germ_layer_palette, stage_palette, type_palette
from devmap.config import phase_palette
from devmap.utils import load_data, save_plot

set_theme()
base_path, plots_path, results_path = get_paths("embedding")

## Load data

In [ ]:
tdata = load_data("umap")
cell_types = pd.read_csv(base_path / "data" / "cell_types.csv", index_col=0)
cluster_umaps = pd.read_csv(results_path / "cluster_umaps.csv", index_col=0)

## Cell types

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
tdata.obs["id"] = tdata.obs["cell_type"].map(cell_types["id"])
sc.pl.umap(tdata, color = "id", legend_fontsize=6, ax = ax,sort_order=False,
           legend_fontweight="regular",palette=celltype_palette, s = .2, 
            legend_loc="on data", frameon=False, title = "", show=False)
save_plot(plots_path / "umap_id.svg", fig, transparent=True, rasterize=True)

## Germ layer

In [213]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
sc.pl.umap(tdata, color = "germ_layer", ax = ax,sort_order=False, s = .2,frameon=False, title = "",
            palette=germ_layer_palette, show=False, legend_loc = None)
save_plot(plots_path / "umap_germ_layer.svg", fig, transparent=True, rasterize=True)

## Stage

In [38]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
sc.pl.umap(tdata, color = "stage", ax = ax,sort_order=False, s = .2,frameon=False, title = "",
            palette=stage_palette, show=False, legend_loc = None)
save_plot(plots_path / "umap_stage.svg", fig, transparent=True, rasterize=True)

## Donor vs Host

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
sc.pl.umap(tdata, color = "type", ax = ax,s = .2,frameon=False, title = "", 
            sort_order=True, show=False, palette=type_palette, legend_loc = None)
save_plot(plots_path / "umap_type.svg", fig, transparent=True, rasterize=True)

## Phase

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=600)
sc.pl.umap(tdata, color = "phase", ax = ax,s = .2,frameon=False, title = "", 
    sort_order=True, show=False, palette=phase_palette, legend_loc = None)
save_plot(plots_path / "umap_phase.svg", fig, transparent=True, rasterize=True)

## Cluster UMAPs

In [ ]:
for cluster in tdata.obs["cluster"].cat.categories:
    cluster_tdata = tdata[tdata.obs.query(f"cluster == @cluster").sample(frac = 1).index,[]].copy()
    cluster_tdata.obsm["X_umap"] = cluster_umaps.loc[cluster_tdata.obs_names, ["UMAP1", "UMAP2"]].values
    fig, ax = plt.subplots(figsize=(2, 2), dpi = 600)
    sc.pl.umap(cluster_tdata, color = "stage", title="",frameon=False,
            legend_loc=None,palette=stage_palette, s = .4, ax = ax, sort_order = False)
    save_plot(plots_path / f"{cluster}_stage_umap.svg", fig, rasterize = True)
    fig, ax = plt.subplots(figsize=(2, 2), dpi = 600)
    sc.pl.umap(cluster_tdata, color = "subtype_id", title="",frameon=False,legend_fontsize=6, legend_fontweight='regular',
            legend_loc="on data",palette = cell_types.set_index("subtype_id")["subtype_color"].to_dict(), s = .4, ax = ax, sort_order = False)
    save_plot(plots_path / f"{cluster}_subtype_umap.svg", fig, rasterize = True)